# Score run H on every calibration and test set (Colab)
Runtime → Change runtime type → **T4 GPU**.

Upload these three files to the Files sidebar and wait for each upload to finish:
- `fallacy_code.zip` (code, tiny)
- `h_adapter.zip` (run H adapter, about 140 MB)
- `external_eval.zip` (CoCoLoFa and MAFALDA test files: **private, local use only, do not share**)

Also needed: the Colab secret `HF_TOKEN` with *Notebook access* on (a Hugging Face read token). The generated sets come from the private dataset `sreejithb/fallacy-dataset` (version `v4`).

Output: one `scores_H_<set>.json` per set (the log-probability of every one of the 186 names for every text), downloaded as `scores_H.zip`. Takes about 20-30 minutes.

In [ ]:
!nvidia-smi -L
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/h_adapter.zip /content/external_eval.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!python fetch_data.py --rev v4
!ls -R out external | head -20
!wc -l data/generated/*.jsonl external/*.jsonl

## Smoke test: 3 texts per set (must finish quickly and print 186 candidates with None included)

In [ ]:
!python score_candidates.py --adapter out/lora_H --run SMOKE --limit 3 --set eval=data/generated/eval.jsonl --set mafalda=external/mafalda_test.jsonl

## Full run

In [ ]:
!python score_candidates.py --adapter out/lora_H --run H \
  --set eval=data/generated/eval.jsonl \
  --set eval_none=data/generated/eval_none.jsonl \
  --set calib_fallacy=data/generated/calib_fallacy.jsonl \
  --set calib_none=data/generated/calib_none.jsonl \
  --set cocolofa_calib=external/cocolofa_calib.jsonl \
  --set cocolofa_test=external/cocolofa_test.jsonl \
  --set mafalda_test=external/mafalda_test.jsonl

In [ ]:
!zip -q scores_H.zip data/scores_H_*.json
from google.colab import files
files.download('/content/work/scores_H.zip')